In [0]:
%pip install dotenv openpyxl 

In [0]:
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.functions import countDistinct
import plotly.express as px
import warnings
from dotenv import load_dotenv
import os
from pathlib import Path

warnings.simplefilter(action='ignore')
load_dotenv()

In [0]:
proj_master_csv = os.getenv("PROJECT_MASTER_CSV_URL")

In [0]:
sel_projects = 'all'

# sel_hierarchy can be 'decentralization' or 'pfm' or 'asset management'
sel_hierarchy = 'hgsf'

#selected countries country_selected can either be 'all' or a list of countries ['Country1','Country']
country_selected = 'all'

#minimum and maximum year to be filtered
min_year = 2018
max_year = 2025

In [0]:

hierarchy_hgsf = {'Key Terms':['school feeding', 'local food procurement', 'procurement for schools','nutrition-sensitive agriculture'], 'Additional': ['home-grown school feeding', 'school meal', 'school feeding', 'school gardens', 'decentralized procurement', 'institutional markets','school management committees', 'farmer cooperatives']}


hierarchy_tagging = {'hgsf':hierarchy_hgsf}
hierarchy = hierarchy_tagging[sel_hierarchy]
print(hierarchy)

search_term_list = []
for each_group in hierarchy:
  search_term_list.append(hierarchy[each_group])
search_term_list = [item for sublist in search_term_list for item in sublist]

print(search_term_list)

In [0]:
path_prior_actions = 'https://thedocs.worldbank.org/en/doc/ed4dbe34c2cd2555d0d7b28952bddf54-0290032023/original/DPADdatabaseFY22.xlsx'
df_prior_actions = pd.read_excel(path_prior_actions,sheet_name='Prior Actions database')

In [0]:
df_project_master = spark.read.format('csv').option("delimiter",",").option("header","false").load('/Volumes/prd_mega/saiana95/vaiana95/GOAT/PROJECT_MASTER_V3.csv')
df_project_master.createOrReplaceTempView("projMaster")

In [0]:
def remove_empty_top_cells(df, header_row_index):
    df_with_id = df.withColumn("row_id", F.monotonically_increasing_id())
    header_row = df_with_id.orderBy("row_id").limit(header_row_index + 1).collect()[header_row_index]
    new_columns = header_row[:-1]
    df_clean = (
        df_with_id
        .orderBy("row_id")
        .filter(F.col("row_id") > header_row_index)
        .drop("row_id")
    )
    df_clean = df_clean.toDF(*new_columns)
    
    return df_clean

In [0]:
df_project_master = remove_empty_top_cells(df_project_master, 4)
display(df_project_master)

In [0]:
df_project_master_subset = df_project_master.toPandas() 

In [0]:
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_ID']!='P169212']

# df_project_master_subset = df_project_master_subset[df_project_master_subset['LNDNG_INSTR_LONG_NAME'].isin(['Development Policy Lending','Investment Project Financing','Program-for-Results Financing'])]
df_project_master_subset = df_project_master_subset[df_project_master_subset['LEAD_GP_CODE'].isin(['AGR'])] # 'AGF','AGR'

# df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_STAT_NAME'].isin(['Closed','Active'])]

if (country_selected=='all'):
  df_project_master_subset = df_project_master_subset
else:
  df_project_master_subset = df_project_master_subset[df_project_master_subset['CNTRY_SHORT_NAME'].isin(country_selected)] 


df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(str)!='None']
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)>=min_year]
df_project_master_subset = df_project_master_subset[df_project_master_subset['PROJ_APPRVL_FY'].astype(int)<=max_year]

In [0]:
df_project_master_subset['PROJ_ID'].nunique()

In [0]:
display(df_project_master_subset)

In [0]:
hierarchy

In [0]:
df_pdo_search = df_project_master_subset[['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC', 'PROJ_APPRVL_FY', 'CNTRY_SHORT_NAME', 'PROJ_DISPLAY_NAME', 'PROJ_ABSTRACT_TEXT', 'LEAD_GP_CODE', 'LEAD_GP_NAME']]

#This is the function to change to incorporate new ML based solution for better categorization
def check_pfm_categories(x,category):
  if(x!=None):
    category_words = hierarchy[category]
    res = any(ele in x for ele in category_words)  
    if(res==True):
      return 'Yes'
    else:
      return None
  else:
    return None

for each_group in hierarchy:
  df_pdo_search[each_group] = df_pdo_search['PROJ_ABSTRACT_TEXT'].apply(check_pfm_categories,category=each_group)

df_pdo_search_selected = df_pdo_search.set_index(['PROJ_ID','PROJ_DEV_OBJECTIVE_DESC', 'PROJ_ABSTRACT_TEXT']).dropna(how='all').reset_index()

display(df_pdo_search_selected.astype(str))

In [0]:
merged_df = pd.merge(df_pdo_search_selected, df_project_master_subset,
                     on=['PROJ_ID', 'PROJ_DEV_OBJECTIVE_DESC']
                    ).dropna(subset=['Key Terms', 'Additional'], how='all')

In [0]:
print(merged_df.shape)
display(merged_df)